**Reading Silver Data**

In [0]:

df_silver = spark.sql("SELECT * FROM parquet.`abfss://silver@carshrudatalake.dfs.core.windows.net/carsales`")

df_silver.display()

** Reading all the dims

In [0]:
df_dealer = spark.sql("SELECT * FROM cars_catalog.gold.dim_dealer")

df_branch = spark.sql("SELECT * FROM cars_catalog.gold.dim_branch")

df_model = spark.sql("SELECT * FROM cars_catalog.gold.dim_model")

df_date = spark.sql("SELECT * FROM cars_catalog.gold.dim_date")


**Bringing Keya to the FACT table**

In [0]:
df_fact = df_silver.join(df_branch, df_silver.branch_ID == df_branch.branch_ID, "left")\
        .join(df_dealer, df_silver.dealer_ID == df_dealer.dealer_ID, "left")\
        .join(df_model, df_silver.Model_ID == df_model.Model_ID, "left")\
        .join(df_date, df_silver.Date_ID == df_date.Date_ID, "left")\
        .select(df_silver.Revenue, df_silver.Units_Sold, df_silver.RevPerUnit, df_branch.dim_branch_key, df_dealer.dim_dealer_key, \
            df_model.dim_model_key,df_date.dim_Date_key)

In [0]:
df_fact.display()

### Writing Fact Table

In [0]:
from delta.tables import DeltaTable

In [0]:
if spark.catalog.tableExists('cars_catalog.gold.factsales'):
    deltatbl = DeltaTable.forName(spark, 'cars_catalog.gold.factsales')
    deltatbl.alias("trg").merge(
        df_fact.alias("src"),"trg.dim_branch_key = src.dim_branch_key AND trg.dim_dealer_key = src.dim_dealer_key\
            and trg.dim_model_key = src.dim_model_key AND trg.dim_Date_key = src.dim_Date_key")\
        .whenMatchedUpdateAll()\
        .whenNotMatchedInsertAll()\
        .execute()

# this is our initial run
else: 
    df_fact.write.format("delta")\
            .mode('overwrite')\
            .option("path",'abfss://gold@carshrudatalake.dfs.core.windows.net/factsales')\
            .saveAsTable("cars_catalog.gold.factsales")

In [0]:
%sql
select * from cars_catalog.gold.factsales